# Piper Neural Voice Training & Fine-Tuning
Use this notebook with a free Google Colab GPU (T4) to train or fine-tune custom Piper TTS voices (like **Tony**, Jarvis, or your own voice) from a dataset of `.wav` files and `metadata.csv`.

### Dataset Structure Required:
```
dataset/
├── metadata.csv
└── wav/
    ├── 0001.wav
    ├── 0002.wav
    └── ...
```
**`metadata.csv` Format:**
```csv
0001|First sentence transcription.
0002|Second sentence transcription.
```

In [ ]:
# Step 1: Install Dependencies & Build Piper Training Tools
!apt-get update -qq && apt-get install -y -qq espeak-ng
!git clone https://github.com/rhasspy/piper.git
%cd piper/src/python
!pip install --upgrade pip
!pip install -e .
!bash build_monotonic_align.sh
%cd ../../..

In [ ]:
# Step 2: Download Base Checkpoint for Fine-Tuning
!mkdir -p checkpoints
!wget -c https://huggingface.co/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt -O checkpoints/base_medium.ckpt

In [ ]:
# Step 3: Upload your Dataset (or Zip file)
# Set your voice name here:
VOICE_NAME = "tony"

from google.colab import files
import os, zipfile

print("Please upload your dataset.zip containing metadata.csv and wav/:")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print("Extracted dataset successfully!")

# If dataset extracted to a subdirectory, find it
dataset_dir = 'dataset'
if not os.path.exists('dataset/metadata.csv'):
    for root, dirs, filenames in os.walk('dataset'):
        if 'metadata.csv' in filenames:
            dataset_dir = root
            break
print(f"Using dataset directory: {dataset_dir}")

In [ ]:
# Step 4: Preprocess Dataset
!python3 -m piper_train.preprocess \
    --language en-us \
    --input-dir {dataset_dir} \
    --output-dir training_dir \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050

In [ ]:
# Step 5: Fine-tune the Neural Voice Model
# 1000 epochs with GPU on a small dataset typically takes 5-10 minutes
!python3 -m piper_train \
    --dataset-dir training_dir \
    --accelerator gpu \
    --devices 1 \
    --batch-size 4 \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs 1000 \
    --resume_from_checkpoint checkpoints/base_medium.ckpt \
    --checkpoint-epochs 25 \
    --precision 32

In [ ]:
# Step 6: Export Trained Model to ONNX
import glob, os, shutil
from google.colab import files

checkpoints = glob.glob("training_dir/lightning_logs/version_*/checkpoints/*.ckpt")
if not checkpoints:
    raise FileNotFoundError("No training checkpoint found!")
latest_ckpt = max(checkpoints, key=os.path.getmtime)
print(f"Exporting checkpoint: {latest_ckpt}")

os.makedirs("exported_model", exist_ok=True)
onnx_path = f"exported_model/{VOICE_NAME}.onnx"
json_path = f"exported_model/{VOICE_NAME}.onnx.json"

!python3 -m piper_train.export_onnx {latest_ckpt} {onnx_path}
shutil.copy2("training_dir/config.json", json_path)

print("Training complete! Downloading exported model and config:")
files.download(onnx_path)
files.download(json_path)
print(f"Place both downloaded files into: central_communication_system/voice_system/models/{VOICE_NAME}/")